# Employee Data Cleaning and Visualization

This notebook uses a clear, step-by-step workflow to clean an employee dataset and explore it with visualizations. Each section explains the purpose of the code before it runs.

**Input:** `sample_data_cleaning_project - Sample_data_cleaning_project.csv`  
**Output:** `cleaned_data.csv`  
**Tools:** Python, pandas, and Matplotlib

## Notebook roadmap

1. Load and inspect the raw dataset
2. Identify and treat missing values
3. Remove exact duplicate rows
4. Detect and remove salary outliers with the IQR method
5. Convert data types and encode department categories
6. Validate and save the cleaned dataset
7. Create four clearly labelled visualizations

## Step 1: Load the Dataset

Load the raw CSV file and confirm that it exists and contains records. Keeping the raw data separate allows the cleaning process to be reproduced.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

INPUT_FILE = Path('sample_data_cleaning_project - Sample_data_cleaning_project.csv')
OUTPUT_FILE = Path('cleaned_data.csv')
PLOT_DIR = Path('visualizations')
PLOT_DIR.mkdir(exist_ok=True)

if not INPUT_FILE.exists():
    raise FileNotFoundError(f'Dataset not found: {INPUT_FILE}')

data = pd.read_csv(INPUT_FILE)
if data.empty:
    raise ValueError('The dataset is empty.')

print(f'Raw dataset shape: {data.shape}')
display(data.head())

## Step 2: Identify Missing Values

Before changing the data, measure missing values in every column. This gives us a baseline for checking whether the treatment step was successful.

In [ ]:
missing_before = data.isna().sum()
print('Missing values before cleaning:')
display(missing_before.to_frame('missing_count'))
print(f'Total missing values: {int(missing_before.sum())}')

## Step 3: Clean Text and Handle Missing Values

Text values are stripped and standardized. Numeric missing values are filled with their median because the median is less affected by unusually high or low salaries. Invalid dates are removed because a joining date cannot be safely guessed.

In [ ]:
# Clean whitespace and standardize text labels.
for column in data.select_dtypes(include='object').columns:
    data[column] = data[column].astype('string').str.strip()

data['Name'] = data['Name'].str.title()
data['Department'] = data['Department'].str.title()

# Convert numeric columns safely before imputing missing values.
data['Age'] = pd.to_numeric(data['Age'], errors='coerce')
data['Salary'] = pd.to_numeric(data['Salary'], errors='coerce')

# Fill missing numeric values with the corresponding column median.
for column in ['Age', 'Salary']:
    median_value = data[column].median()
    if pd.isna(median_value):
        raise ValueError(f'No valid values available for {column}.')
    data[column] = data[column].fillna(median_value)

# Parse dates and remove records whose date is invalid or missing.
data['Join_Date'] = pd.to_datetime(data['Join_Date'], errors='coerce')
invalid_dates = int(data['Join_Date'].isna().sum())
data = data.dropna(subset=['Join_Date']).copy()

print('Missing values after treatment:')
display(data.isna().sum().to_frame('missing_count'))
print(f'Invalid/missing dates removed: {invalid_dates}')

## Step 4: Check and Remove Duplicate Rows

Only exact duplicate records are removed. Employees with the same name or age are not automatically considered duplicates because they may be different people.

In [ ]:
# Count exact duplicates before removing them.
duplicates_before = int(data.duplicated().sum())

# Remove only rows that match in every column.
data = data.drop_duplicates().copy()

print(f'Exact duplicate rows removed: {duplicates_before}')
print(f'Duplicate rows remaining: {int(data.duplicated().sum())}')

## Step 5: Handle Salary Outliers

The interquartile range (IQR) method identifies unusually low or high salaries. Values outside `Q1 - 1.5 × IQR` and `Q3 + 1.5 × IQR` are excluded from the cleaned analysis.

In [ ]:
# Calculate the first quartile, third quartile, and IQR.
q1 = data['Salary'].quantile(0.25)
q3 = data['Salary'].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

# Keep salaries inside the accepted IQR range.
outlier_mask = ~data['Salary'].between(lower_bound, upper_bound)
outliers_removed = int(outlier_mask.sum())
data = data.loc[~outlier_mask].copy()

print(f'Q1: {q1:.2f}')
print(f'Q3: {q3:.2f}')
print(f'IQR: {iqr:.2f}')
print(f'Lower bound: {lower_bound:.2f}')
print(f'Upper bound: {upper_bound:.2f}')
print(f'Salary outliers removed: {outliers_removed}')

## Step 6: Convert Data Types

Use suitable data types so that ages, salaries, and dates can be analysed correctly. Converting age to an integer also makes the final output easier to read.

In [ ]:
data['Age'] = data['Age'].round().astype(int)
data['Salary'] = data['Salary'].astype(float)
data['Join_Date'] = pd.to_datetime(data['Join_Date'])

print('Final data types before encoding:')
print(data.dtypes)

## Step 7: Encode Categorical Variables

Machine-learning and statistical workflows need numeric representations of categories. One-hot encoding creates one indicator column for each department and keeps all department categories visible.

In [ ]:
# Convert Department into binary indicator columns.
data = pd.get_dummies(
    data,
    columns=['Department'],
    prefix='Department',
    drop_first=False,
    dtype=int
)

department_columns = [c for c in data.columns if c.startswith('Department_')]
print('Encoded department columns:')
print(department_columns)
display(data.head())

## Step 8: Validate the Cleaned Dataset

Validation checks confirm that the cleaning rules worked: the output is not empty, missing values are gone, and exact duplicates do not remain.

In [ ]:
print(f'Final dataset shape: {data.shape}')
print('Missing values:')
print(data.isna().sum())
print(f'Duplicate rows: {int(data.duplicated().sum())}')

if data.empty:
    raise ValueError('Validation failed: no records remain.')
if data.isna().sum().sum() != 0:
    raise ValueError('Validation failed: missing values remain.')
if data.duplicated().any():
    raise ValueError('Validation failed: duplicate rows remain.')

print('Validation successful: dataset is clean.')

## Step 9: Save the Cleaned Dataset

Export the validated dataframe so it can be reused without repeating the cleaning steps.

In [ ]:
data.to_csv(OUTPUT_FILE, index=False)
if not OUTPUT_FILE.exists():
    raise IOError('The cleaned dataset was not saved successfully.')
print(f'Cleaned dataset saved successfully as: {OUTPUT_FILE}')

## Step 10: Load the Cleaned Dataset for Visualization

The charts use the saved cleaned file. A readable department label is reconstructed from the one-hot encoded columns for plotting.

In [ ]:
cleaned_data = pd.read_csv(OUTPUT_FILE, parse_dates=['Join_Date'])
department_columns = [c for c in cleaned_data.columns if c.startswith('Department_')]

def get_department(row):
    for column in department_columns:
        if row[column] == 1:
            return column.replace('Department_', '')
    return 'Unknown'

chart_data = cleaned_data.copy()
chart_data['Department_Label'] = chart_data.apply(get_department, axis=1)
chart_data['Join_Year'] = chart_data['Join_Date'].dt.year
print('Cleaned dataset loaded for visualization.')
display(chart_data.head())

## Visualization 1: Department Distribution — Bar Plot

A bar plot compares the number of employees in each department.

In [ ]:
department_counts = chart_data['Department_Label'].value_counts().sort_values(ascending=False)
plt.figure(figsize=(8, 5))
plt.bar(department_counts.index, department_counts.values, color='#4C78A8')
plt.title('Department Distribution')
plt.xlabel('Department')
plt.ylabel('Number of Employees')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'department_distribution_bar.png', dpi=150, bbox_inches='tight')
plt.show()

## Visualization 2: Salary Distribution — Histogram

A histogram shows how frequently salary values occur across salary ranges.

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(chart_data['Salary'], bins=10, color='#F58518', edgecolor='black')
plt.title('Salary Distribution')
plt.xlabel('Salary')
plt.ylabel('Frequency')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'salary_distribution_histogram.png', dpi=150, bbox_inches='tight')
plt.show()

## Visualization 3: Salary vs. Age — Scatter Plot

A scatter plot helps inspect whether salary and age show an obvious relationship. Each point represents one employee.

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(chart_data['Age'], chart_data['Salary'], color='#54A24B', alpha=0.8, edgecolor='black')
plt.title('Salary vs. Age')
plt.xlabel('Age')
plt.ylabel('Salary')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'salary_vs_age_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## Visualization 4: Salary vs. Age by Department — Grouped Scatter Plot

The grouped scatter plot uses a different colour for each department, making it easier to compare age and salary patterns across groups.

In [ ]:
plt.figure(figsize=(9, 6))
for department, group in chart_data.groupby('Department_Label'):
    plt.scatter(group['Age'], group['Salary'], label=department, alpha=0.8, s=70)

plt.title('Salary vs. Age by Department')
plt.xlabel('Age')
plt.ylabel('Salary')
plt.legend(title='Department')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'salary_vs_age_by_department_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Four visualizations saved in: {PLOT_DIR.resolve()}')

## Conclusion

The dataset is now cleaned, validated, exported, and visualized. The comments and section headings document why each operation is performed, including missing-value treatment, duplicate removal, outlier handling, type conversion, and categorical encoding.